# Sommelier — Vietnamese pipeline on a single A100

Ported from `19-8-sommerlier(1).ipynb` (Kaggle, 2x T4). Same four isolated `uv`
venvs and the same install order; what changes is where things live and how the
GPUs are addressed.

**What is different from the Kaggle notebook**

| | Kaggle 2x T4 | This notebook |
|---|---|---|
| GPUs | 2 x 16GB, workers split across them | 1 x A100 40/80GB, all workers share it |
| Profile | `--env kaggle` | `--env a100` |
| Precision | float16 (Turing has no bf16) | **bfloat16**, flash-attention on |
| Paths | `/kaggle/working`, `/kaggle/temp` | detected below, works on Colab or a bare VM |
| Batch cap | 5h of audio per run | 20h |

**Read this before running.** The `a100` profile sets
`diarizen.segmentation_step = 0.5` where `kaggle` uses `0.1` — a 8s hop instead
of 1.6s over the same 16s window. That is 5x coarser, and it lands directly on
the short backchannels this pipeline exists to catch. Results from the two
profiles are not just faster or slower, they are **different**. Section 8 checks
what it cost you. If backchannel counts drop, set it back to 0.1 in
`config.json` — an A100 can afford the extra compute that Kaggle could not.

## 0. Where are we running?

In [ ]:
import os, sys, subprocess, pathlib

# Colab mounts /content; a bare VM or a rented box just uses the cwd. Nothing
# here assumes Kaggle's /kaggle/working + /kaggle/temp split.
if os.path.exists("/content"):
    BASE_DIR, RUNTIME = "/content", "colab"
else:
    BASE_DIR, RUNTIME = os.getcwd(), "generic"

# Kaggle kept venvs on a separate volume for quota reasons. Here one root is
# simpler, and keeping the venvs beside the repo means one directory to delete.
TEMP_DIR    = os.path.join(BASE_DIR, "sommelier_envs")
PROJECT_DIR = os.path.join(BASE_DIR, "sommerlier")
ENV_DIR     = os.path.join(TEMP_DIR, "sommelier_env")
AUDIO_DIR   = os.path.join(BASE_DIR, "vi_audio")

for d in (TEMP_DIR, AUDIO_DIR):
    os.makedirs(d, exist_ok=True)

os.environ["MPLBACKEND"] = "Agg"
os.environ["UV_CACHE_DIR"] = os.path.join(BASE_DIR, ".uv_cache")
os.environ["UV_LINK_MODE"] = "copy"

print(f"runtime     {RUNTIME}")
for k, v in [("BASE_DIR", BASE_DIR), ("PROJECT_DIR", PROJECT_DIR),
             ("TEMP_DIR", TEMP_DIR), ("AUDIO_DIR", AUDIO_DIR)]:
    print(f"{k:12s}{v}")

## 1. Check the GPU is actually an A100

In [ ]:
import subprocess, sys

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("python", sys.version.split()[0])

q = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip().splitlines()

for i, line in enumerate(q):
    print(f"  GPU {i}: {line}")

name, mem, cap = [x.strip() for x in q[0].split(",")]
cap_major = int(float(cap))

if cap_major < 8:
    print(f"\n!! compute capability {cap} is pre-Ampere. bfloat16 and flash-attention")
    print("   are unavailable, so the a100 profile will fail or fall back silently.")
    print("   Use --env kaggle instead.")
elif "A100" not in name:
    print(f"\n note: {name} is Ampere-or-newer so the profile will work, but the")
    print("   batch sizes in it were tuned for A100 memory. Watch for OOM.")

# The profile puts every worker on gpu 0. With more than one GPU visible the
# workers would still all land on 0 and leave the rest idle.
if len(q) > 1:
    print(f"\n note: {len(q)} GPUs visible. The a100 profile sets gpu_1 = gpu_2 = 0,")
    print("   so only the first is used. Set CUDA_VISIBLE_DEVICES to pick which one.")

## 2. Clone the repository

In [ ]:
import os, shutil

REPO   = "https://github.com/foresst123/sommerlier.git"
BRANCH = "solid-architecture"

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

!git clone -q --branch {BRANCH} {REPO} {PROJECT_DIR}
!cd {PROJECT_DIR} && git log --oneline -1

## 3. Four isolated venvs (~10 min)

Same split as the Kaggle notebook, and for the same reason: DiariZen needs a
patched pyannote, Qwen3-ASR needs a newer huggingface-hub than WhisperX
tolerates, and Sidon needs `--no-build-isolation` so flash-attn can see torch.
One environment cannot satisfy all three.

In [ ]:
!pip install -q uv

!uv venv --allow-existing {ENV_DIR} --python 3.12

REQUIREMENTS = '''
numpy==2.2.2
torch==2.8.0
torchaudio==2.8.0
torchvision==0.23.0
lightning==2.4.0
torchmetrics==1.6.2
onnxruntime-gpu==1.19.2
pyannote.audio==4.0.7
speechbrain==1.0.2
faster-whisper==1.2.0
whisperx==3.8.6
ctranslate2==4.5.0
demucs>=4.0.1
panns-inference
librosa==0.10.2.post1
soundfile==0.13.1
pydub==0.25.1
julius==0.2.7
numba==0.61.2
transformers==4.53.0
huggingface-hub>=0.9.8
openai==1.63.0
pandas==2.2.3
PyYAML==6.0.2
tqdm==4.67.1
requests==2.32.4
einops==0.8.1
hydra-core==1.3.2
omegaconf==2.3.0
setuptools==75.0.0
accelerate==0.30.0
protobuf
'''
req_file = os.path.join(TEMP_DIR, "requirements.txt")
open(req_file, "w").write(REQUIREMENTS.strip())

print(">>> torch first, so everything else resolves against it")
!uv pip install --python {ENV_DIR} torch==2.8.0 torchaudio==2.8.0 torchvision==0.23.0 --extra-index-url https://download.pytorch.org/whl/cu126

print("\n>>> the rest")
!uv pip install --python {ENV_DIR} -r {req_file} --extra-index-url https://download.pytorch.org/whl/cu126 --index-strategy unsafe-best-match

In [ ]:
# --- Sidon (separation) ---
SIDON_ENV_DIR = os.path.join(TEMP_DIR, "sidon_env")

!uv venv --allow-existing {SIDON_ENV_DIR} --python 3.12
!uv pip install --python {SIDON_ENV_DIR} torch==2.8.0 torchaudio==2.8.0 torchvision==0.23.0 --extra-index-url https://download.pytorch.org/whl/cu126
!uv pip install --python {SIDON_ENV_DIR} hatchling packaging ninja

# --no-build-isolation so flash-attn's build can import the torch just installed.
!uv pip install --python {SIDON_ENV_DIR} --no-build-isolation "git+https://github.com/sarulab-speech/Sidon.git@c8cde2b24e4c77c599ad43a9871140cdc9beeffa"

!{SIDON_ENV_DIR}/bin/python -c "from sidon.model.dialogue_sidion.lightning_module import DialogueSidonDiffusionLightningModule; print('Sidon env OK')"

In [ ]:
# --- Qwen3-ASR ---
QWEN3_ENV_DIR = os.path.join(TEMP_DIR, "qwen3_env")

!uv venv --allow-existing {QWEN3_ENV_DIR} --python 3.12
!uv pip install --python {QWEN3_ENV_DIR} torch==2.8.0 torchaudio==2.8.0 torchvision==0.23.0 --extra-index-url https://download.pytorch.org/whl/cu126
!uv pip install --python {QWEN3_ENV_DIR} "transformers>=5.13.0" "huggingface-hub>=1.5.0" accelerate soundfile librosa

# The a100 profile turns flash-attention on for this worker; Turing could not.
!uv pip install --python {QWEN3_ENV_DIR} --no-build-isolation flash-attn --no-cache-dir || echo "flash-attn build failed -- set models.qwen3.use_flash_attention=false in config.json"

!{QWEN3_ENV_DIR}/bin/python -c "from transformers import AutoProcessor, AutoModelForMultimodalLM; print('Qwen3 env OK')"

In [ ]:
# --- DiariZen (needs its own patched pyannote) ---
DIARIZEN_ENV_DIR = os.path.join(TEMP_DIR, "diarizen_env")
DZ = f"{DIARIZEN_ENV_DIR}/bin/python"
PIN = "844f5555b0a98acd0931511fc641a8c5b8ba92c7"

!rm -rf {DIARIZEN_ENV_DIR}
!uv venv {DIARIZEN_ENV_DIR} --python 3.12

!uv pip install --python {DZ} torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu126
!uv pip install --python {DZ} numpy==1.26.4 scipy==1.13.1 pandas==2.2.3 numba==0.59.1 llvmlite==0.42.0
!uv pip install --python {DZ} soundfile librosa==0.10.2.post1 matplotlib==3.9.4 pyparsing accelerate==1.12.0
!uv pip install --python {DZ} "git+https://github.com/BUTSpeechFIT/DiariZen.git@{PIN}#subdirectory=pyannote-audio"
!uv pip install --python {DZ} git+https://github.com/BUTSpeechFIT/DiariZen.git@{PIN}
!uv pip install --python {DZ} speechbrain==1.0.2 toml==0.10.2 wrapt==2.3.0 psutil

!uv cache clean

## 4. Hugging Face token

In [ ]:
import json, os, pathlib, getpass

token = os.environ.get("HF_TOKEN", "")
if not token:
    try:            # Colab keeps secrets in a userdata store
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not token:
    token = getpass.getpass("HF token (needs pyannote gated-model access): ").strip()

os.environ["HF_TOKEN"] = token
os.environ["HUGGINGFACE_HUB_TOKEN"] = token

cfg_path = pathlib.Path(PROJECT_DIR) / "podcast-pipeline" / "config.json"
cfg = json.loads(cfg_path.read_text())
cfg["huggingface_token"] = token
cfg_path.write_text(json.dumps(cfg, indent=2, ensure_ascii=False))
print("token written to config.json")

## 5. Audio in

Drop files into `AUDIO_DIR`. Two files that share a stem (`talk.mp3` and
`talk.wav`) resolve to one output directory, so the pipeline refuses the run
rather than overwriting one with the other -- the check below catches it here
instead of after the models have loaded.

In [ ]:
import glob, os, collections

exts = ("*.mp3", "*.wav", "*.m4a", "*.flac", "*.ogg")
files = sorted(f for e in exts for f in glob.glob(os.path.join(AUDIO_DIR, e)))

if not files:
    print(f"No audio in {AUDIO_DIR}. Upload some and re-run this cell.")
    if RUNTIME == "colab":
        from google.colab import files as colab_files
        for name, data in colab_files.upload().items():
            open(os.path.join(AUDIO_DIR, name), "wb").write(data)
        files = sorted(f for e in exts for f in glob.glob(os.path.join(AUDIO_DIR, e)))

stems = collections.defaultdict(list)
for f in files:
    stems[os.path.splitext(os.path.basename(f))[0]].append(os.path.basename(f))
clash = {s: g for s, g in stems.items() if len(g) > 1}

total = 0.0
for f in files:
    try:
        import soundfile as sf
        info = sf.info(f)
        total += info.duration
        print(f"  {os.path.basename(f):45s} {info.duration/60:6.1f} min  {info.samplerate} Hz")
    except Exception:
        print(f"  {os.path.basename(f):45s}  (duration unknown)")

print(f"\n{len(files)} file(s), {total/3600:.2f} h total")
if clash:
    print("\n!! same stem, would collide in the output directory:")
    for s, g in clash.items():
        print(f"   {s}: {', '.join(g)}")

## 6. Run

`--env a100` selects the profile: bfloat16 everywhere, flash-attention on,
larger batches, both worker slots pinned to GPU 0, and a 20h batch cap.

`--by_stage` runs one stage across every file before the next, so each model
loads once per batch instead of once per file. With `--keep_models` (already on
in the a100 profile) the saving compounds.

In [ ]:
import os, sys

os.chdir(os.path.join(PROJECT_DIR, "podcast-pipeline"))
python_bin = os.path.join(ENV_DIR, "bin", "python")

site_packages = os.path.join(ENV_DIR, "lib",
                             f"python{sys.version_info.major}.{sys.version_info.minor}",
                             "site-packages")
os.environ["LD_LIBRARY_PATH"] = (os.environ.get("LD_LIBRARY_PATH", "") +
    f":{site_packages}/nvidia/cudnn/lib:{site_packages}/torch/lib")

# Uncomment to skip the LLM refinement pass. It is the slowest stage by far and
# it does not touch anything the separation diagnostics below report on.
# EXTRA = ""
EXTRA = "--llm_refinement"

!CUDA_VISIBLE_DEVICES=0 {python_bin} main.py \
    --audio_dir "{AUDIO_DIR}" \
    --env a100 \
    --lang vi \
    --tse --panns --vad --ASRMoE --by_stage \
    {EXTRA}

## 7. Results, in the browser

Everything below reads what the run wrote to disk. Nothing recomputes.

In [ ]:
import glob, json, os
import pandas as pd
from IPython.display import display, HTML, Audio

pd.set_option("display.max_colwidth", 140)

runs = sorted(glob.glob(os.path.join(AUDIO_DIR, "_final", "*", "*")))
runs = [r for r in runs if os.path.isdir(r)]
if not runs:
    runs = sorted(p for p in glob.glob(os.path.join(AUDIO_DIR, "_final", "*")) if os.path.isdir(p))

print(f"{len(runs)} output folder(s):")
for r in runs:
    print("  ", r)

OUT = runs[-1] if runs else None
print(f"\nshowing: {OUT}")

### 7a. Transcript

In [ ]:
import os, json, glob
from IPython.display import display, HTML
import pandas as pd

js = glob.glob(os.path.join(OUT, "*.json"))
js = [p for p in js if "intermediate" not in p and "tse_report" not in p]

if not js:
    print("no transcript json here")
else:
    data = json.load(open(js[0], encoding="utf-8"))
    segs = data.get("segments", data) if isinstance(data, dict) else data
    df = pd.DataFrame(segs)
    keep = [c for c in ("index", "start", "end", "speaker", "text") if c in df.columns]
    df = df[keep]
    df["dur"] = (df["end"] - df["start"]).round(2)
    display(HTML(f"<b>{os.path.basename(js[0])}</b> — {len(df)} segments"))
    display(df.head(60))

### 7b. Listen to the whole file, and to each segment

In [ ]:
import glob, os
from IPython.display import display, HTML, Audio

src = [f for f in files if os.path.splitext(os.path.basename(f))[0] == os.path.basename(OUT)]
if src:
    display(HTML("<b>Original</b>"))
    display(Audio(src[0]))

# export_mp3_segments writes into <out>/<audio_name>/
seg_dir = os.path.join(OUT, os.path.basename(OUT))
seg_files = sorted(glob.glob(os.path.join(seg_dir, "*.mp3")) +
                   glob.glob(os.path.join(seg_dir, "*.wav")))

print(f"{len(seg_files)} exported segments in {seg_dir}")
for f in seg_files[:12]:
    display(HTML(f"<code>{os.path.basename(f)}</code>"))
    display(Audio(f))

### 7c. What separation actually did

The three players per row are the point: the mixture the separator was given,
and the two tracks it returned. A wrong speaker assignment is obvious in seconds
here and invisible in any log line.

In [ ]:
import glob, json, os
from IPython.display import display, HTML, Audio
import pandas as pd

rep = glob.glob(os.path.join(OUT, "*_tse_report.json"))
if not rep:
    print("no tse_report.json — was --tse on?")
else:
    r = json.load(open(rep[0], encoding="utf-8"))
    display(HTML("<b>Thresholds in force</b>"))
    display(pd.DataFrame([r["thresholds"]]))
    display(HTML("<b>Counters</b>"))
    display(pd.DataFrame([r["stats"]]).T.rename(columns={0: "count"}))
    if r.get("sim_percentiles"):
        display(HTML("<b>ECAPA similarity percentiles</b>"))
        display(pd.DataFrame([r["sim_percentiles"]]))
    if r.get("failures"):
        display(HTML(f"<b>{len(r['failures'])} discarded span(s)</b>"))
        display(pd.DataFrame(r["failures"]))

# Clips dumped per overlap: *_mix / *_trackA / *_trackB
sep_dir = os.path.join(OUT, "separation")
for sub in ("failed", "kept", ""):
    d = os.path.join(sep_dir, sub) if sub else sep_dir
    mixes = sorted(glob.glob(os.path.join(d, "*_mix.wav")))
    if not mixes:
        continue
    display(HTML(f"<h4>{sub or 'separation'} — {len(mixes)} case(s)</h4>"))
    for m in mixes[:6]:
        tag = os.path.basename(m)[:-8]
        display(HTML(f"<code>{tag}</code>"))
        for label, suffix in (("mixture", "_mix"), ("track A", "_trackA"), ("track B", "_trackB")):
            p = os.path.join(d, tag + suffix + ".wav")
            if os.path.exists(p):
                display(HTML(f"&nbsp;&nbsp;{label}"))
                display(Audio(p))

## 8. Did the a100 profile cost you backchannels?

`segmentation_step` is 0.5 here against 0.1 on Kaggle. The `[DIAR:*]` counters
in the run log are the direct comparison; this cell reads the same thing from
the exported diarization instead.

A drop in sub-0.5s segments is the signal that the coarser step is skipping the
short interjections. If so, set `models.diarizen.segmentation_step` back to
`0.1` in `config.json` and re-run — an A100 has the compute budget that a T4
did not.

In [ ]:
import glob, json, os
import pandas as pd
from IPython.display import display, HTML

rows = []
for run_dir in runs:
    f = glob.glob(os.path.join(run_dir, "*_intermediate_diarization.json"))
    if not f:
        continue
    segs = json.load(open(f[0], encoding="utf-8"))
    d = pd.Series([s["end"] - s["start"] for s in segs])
    rows.append({
        "file": os.path.basename(run_dir),
        "n": len(segs),
        "speakers": len({s["speaker"] for s in segs}),
        "p50_dur": round(d.median(), 2),
        "max_dur": round(d.max(), 2),
        "under_0.5s": int((d < 0.5).sum()),
        "under_0.2s": int((d < 0.2).sum()),
    })

if rows:
    display(HTML("<b>Post-processing diarization shape</b> "
                 "(this is after VAD, merge and split — not raw DiariZen)"))
    display(pd.DataFrame(rows))
    print("\nFor the raw diarizer numbers, grep the run log for '[DIAR:raw]'.")
else:
    print("no *_intermediate_diarization.json found")

### 9. Package the outputs

In [ ]:
import shutil, os
archive = os.path.join(BASE_DIR, "sommelier_output")
shutil.make_archive(archive, "zip", os.path.join(AUDIO_DIR, "_final"))
print(f"{archive}.zip  ({os.path.getsize(archive + '.zip')/1e6:.1f} MB)")

if RUNTIME == "colab":
    from google.colab import files as colab_files
    colab_files.download(archive + ".zip")

## Notes for this hardware

**One GPU, not two.** The a100 profile sets `gpu_1 = gpu_2 = 0`, so the DiariZen
and Qwen3 workers share one device instead of getting one each. On 40GB that is
fine; on a 40GB card running `--keep_models` with the larger batch sizes it is
worth watching `nvidia-smi` on the first run.

**bfloat16 and flash-attention are on.** Both need compute capability 8.0+. If
you are on a V100 or T4 despite the notebook name, use `--env kaggle`.

**Batch cap is 20h, not 5h.** Set `--max_hours` to override, and `--only_batch N`
to run one batch per session if the box has a runtime limit.

**If you hit CUDA OOM:** drop `keep_models` from the profile first (it trades
VRAM for speed), then halve `models.*.batch_size`.

**Untested.** This notebook was written against the Kaggle one and the repo's
`a100` profile, but has not been executed on an actual A100 — no such hardware
was available. The install cells are copied verbatim from the working Kaggle
notebook, so the likely failure points are the two things that differ: the
flash-attn build in the Qwen3 env, and the paths in section 0.